In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats

# Multiple linear regression
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Notebook display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Visualization settings
sns.set_theme(style='whitegrid')

print("All libraries imported successfully!")

In [ ]:
# Import supermarket sales dataset
import pandas as pd

df = pd.read_csv('../data/SuperMarket Analysis.csv')

# Display the first five rows
df.head()

In [ ]:
# Examine the structure of our supermarket dataset

# Number of rows and columns
print("Dataset Shape:", df.shape)

# Display all column names
print("\nColumn Names:")
print(df.columns.tolist())

# Check data types
print("\nData Types:")
print(df.dtypes)

# Generate summary statistics
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check for duplicate transactions
print("\nDuplicate Rows:", df.duplicated().sum())

# Check the number of unique values in each column
print("\nUnique Values:")
print(df.nunique())

In [ ]:
# FEATURE ENGINEERING

# Convert transaction dates into datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Create a numerical variable for customer membership
# 1 = Member, 0 = Normal customer
df['Member'] = (df['Customer type'] == 'Member').astype(int)

# Extract the month from the transaction date
df['Month'] = df['Date'].dt.month

# Extract the day of the week
df['Day_of_Week'] = df['Date'].dt.day_name()

# Display our new variables
df[['Customer type', 'Member', 'Date', 'Month', 'Day_of_Week']].head(10)

In [ ]:
# Verify customer membership encoding
print("Membership Counts:")
print(df['Member'].value_counts())

# Check transaction counts by month
print("\nTransactions by Month:")
print(df['Month'].value_counts().sort_index())

# Check transaction counts by weekday
print("\nTransactions by Day:")
print(df['Day_of_Week'].value_counts())

In [ ]:
# EDA 1: Customer Purchasing Behavior

# Calculate average transaction value by customer type
avg_sales = df.groupby('Customer type')['Sales'].mean()

print("Average Transaction Value by Customer Type:")
print(avg_sales.round(2))

# Calculate total revenue by customer type
total_sales = df.groupby('Customer type')['Sales'].sum()

print("\nTotal Revenue by Customer Type:")
print(total_sales.round(2))

# Count transactions by customer type
customer_count = df['Customer type'].value_counts()

print("\nTransactions by Customer Type:")
print(customer_count)

In [ ]:
# EDA 2: Member vs Nonmember Spending by Product Category

# Calculate average transaction value by product and customer type
category_spending = df.pivot_table(
    values='Sales',
    index='Product line',
    columns='Customer type',
    aggfunc='mean'
)

# Calculate difference in average spending
category_spending['Difference'] = (
    category_spending['Member'] - category_spending['Normal']
)

# Display results
print("Average Spending by Product Category:")
display(category_spending.round(2))

In [ ]:
# HYPOTHESIS TEST: Member vs Nonmember Spending

from scipy import stats

# Separate transaction values by customer type
member_sales = df[df['Customer type'] == 'Member']['Sales']
normal_sales = df[df['Customer type'] == 'Normal']['Sales']

# Calculate average transaction values
print("Member Average:", round(member_sales.mean(), 2))
print("Nonmember Average:", round(normal_sales.mean(), 2))

# Calculate difference in averages
difference = member_sales.mean() - normal_sales.mean()

print("Average Difference:", round(difference, 2))

# Perform Welch's two-sample t-test
t_stat, p_value = stats.ttest_ind(
    member_sales,
    normal_sales,
    equal_var=False
)

print("\nT-Statistic:", round(t_stat, 4))
print("P-Value:", round(p_value, 4))

# Interpret statistical significance
if p_value < 0.05:
    print("Result: Reject the null hypothesis.")
    print("The difference is statistically significant.")
else:
    print("Result: Fail to reject the null hypothesis.")
    print("The difference is not statistically significant.")

In [ ]:
# MULTIPLE LINEAR REGRESSION
# Dependent variable: Sales
# Independent variables: Membership, Quantity, Product Line, Branch

import statsmodels.formula.api as smf

# Fit the multiple linear regression model
model = smf.ols(
    formula='Sales ~ Member + Quantity + C(Q("Product line")) + C(Branch)',
    data=df
).fit(cov_type='HC3')

# Display regression results
print(model.summary())

In [ ]:
# CATEGORY-SPECIFIC HYPOTHESIS TESTING

from scipy import stats
import pandas as pd

results = []

# Loop through each product category
for category in df['Product line'].unique():

    # Filter transactions by category and membership
    subset = df[df['Product line'] == category]

    members = subset[subset['Customer type'] == 'Member']['Sales']
    nonmembers = subset[subset['Customer type'] == 'Normal']['Sales']

    # Perform Welch's two-sample t-test
    t_stat, p_value = stats.ttest_ind(
        members,
        nonmembers,
        equal_var=False
    )

    # Store results
    results.append({
        'Product Category': category,
        'Member Count': len(members),
        'Nonmember Count': len(nonmembers),
        'Member Average': members.mean(),
        'Nonmember Average': nonmembers.mean(),
        'Difference': members.mean() - nonmembers.mean(),
        'T-Statistic': t_stat,
        'P-Value': p_value
    })

# Convert results into a DataFrame
category_tests = pd.DataFrame(results)

# Sort by spending difference
category_tests = category_tests.sort_values(
    'Difference',
    ascending=False
)

display(category_tests.round(4))

In [ ]:
# MULTIPLE COMPARISON CORRECTION

from statsmodels.stats.multitest import multipletests

# Apply Holm correction to category p-values
reject, adjusted_p, _, _ = multipletests(
    category_tests['P-Value'],
    alpha=0.05,
    method='holm'
)

# Add adjusted results to our table
category_tests['Adjusted P-Value'] = adjusted_p
category_tests['Significant'] = reject

# Display final statistical results
display(
    category_tests[
        [
            'Product Category',
            'Difference',
            'P-Value',
            'Adjusted P-Value',
            'Significant'
        ]
    ].round(4)
)

In [ ]:
# VISUALIZATION: MEMBERSHIP SPENDING DIFFERENCES

plt.figure(figsize=(10, 6))

sns.barplot(
    data=category_tests,
    x='Difference',
    y='Product Category',
    color='steelblue'
)

plt.axvline(0, color='black', linestyle='--')

plt.title('Member vs Nonmember Spending Differences by Category')
plt.xlabel('Difference in Average Transaction Value ($)')
plt.ylabel('Product Category')

plt.tight_layout()
plt.show()

In [ ]:
# Display category-specific hypothesis test results

display(
    category_tests[
        [
            'Product Category',
            'Member Count',
            'Nonmember Count',
            'Difference',
            'P-Value',
            'Adjusted P-Value',
            'Significant'
        ]
    ].round(4)
)

In [ ]:
# VISUALIZATION: Correlation Heatmap

# Select numerical variables
corr_data = df[
    ['Sales', 'Quantity', 'Unit price', 'Rating', 'Member']
]

# Calculate correlation matrix
correlation = corr_data.corr()

# Create heatmap
plt.figure(figsize=(9, 6))

sns.heatmap(
    correlation,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    vmin=-1,
    vmax=1
)

plt.title('Correlation Heatmap: Supermarket Sales')
plt.tight_layout()
plt.show()

In [ ]:
# VISUALIZATION: Quantity vs Sales

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='Quantity',
    y='Sales',
    hue='Customer type',
    alpha=0.6
)

plt.title('Purchase Quantity vs Transaction Value')
plt.xlabel('Quantity Purchased')
plt.ylabel('Transaction Value ($)')
plt.legend(title='Customer Type')

plt.tight_layout()
plt.show()

In [ ]:
# EDA: Membership Spending by Branch

branch_membership = df.pivot_table(
    values='Sales',
    index='Branch',
    columns='Customer type',
    aggfunc='mean'
)

branch_membership['Difference'] = (
    branch_membership['Member'] -
    branch_membership['Normal']
)

display(branch_membership.round(2))

In [ ]:
# EDA: Average Purchase Quantity by Customer Type

quantity_analysis = df.groupby('Customer type')['Quantity'].agg(
    Average_Quantity='mean',
    Median_Quantity='median',
    Transactions='count'
)

display(quantity_analysis.round(2))

# Test whether average quantities differ
member_qty = df[df['Customer type'] == 'Member']['Quantity']
normal_qty = df[df['Customer type'] == 'Normal']['Quantity']

t_stat, p_value = stats.ttest_ind(
    member_qty,
    normal_qty,
    equal_var=False
)

print("T-statistic:", round(t_stat, 4))
print("P-value:", round(p_value, 4))

In [ ]:
# EDA: Average Sales by Day of Week

day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

daily_sales = (
    df.groupby('Day_of_Week')['Sales']
    .mean()
    .reindex(day_order)
)

plt.figure(figsize=(10, 5))

sns.barplot(
    x=daily_sales.index,
    y=daily_sales.values,
    color='steelblue'
)

plt.title('Average Transaction Value by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Transaction Value ($)')
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# EDA: Customer Satisfaction

rating_analysis = df.groupby('Customer type')['Rating'].agg(
    Average_Rating='mean',
    Median_Rating='median',
    Transactions='count'
)

display(rating_analysis.round(2))